# Residual correction distribution

How large are the corrections the **TD3 residual actor** applies on top of the
frozen TFv6 base policy? This notebook histograms the per-frame correction.

**What "correction" means here.** In these runs the route residual is disabled
(`disable_residual_route = True`), so the only active correction is to the
**target speed**:

$$\Delta v = \alpha_{\text{speed}} \cdot c_{\text{speed}} \cdot v_{\max},
\qquad c_{\text{speed}} = \tanh(\cdot) \in [-1, 1].$$

With $\alpha_{\text{speed}} = 0.35$ and $v_{\max} = 25\,\text{m/s}$ the actor can
shift the target speed by at most $\pm 8.75\,\text{m/s}$. A correction of $0$
means the residual leaves the TFv6 base speed untouched.

**One shared set of evaluation states.** Every run is scored on the **same**
pre-encoded states, taken from a single replay buffer (`SHARED_BUFFER`). A saved
buffer stores `obs_bev`, `obs_kv_status_mean`, `obs_base_action_mean` (and
`obs_speed_history` when used), which feed `actor.forward_coeffs_from_features`
directly — so the frozen TFv6 runs **zero** times and any comparison between runs
reflects **only the policy weights**, never a difference in the states. The
returned correction is the deterministic (noise-free) actor output.

Each run only needs its own `model_*.pth` checkpoint; it does **not** need a
buffer of its own. States are stride-sampled evenly across `SHARED_BUFFER` (the
exact same rows for every run) and cached to `.npy` for instant re-plotting.

> Note: `SHARED_BUFFER` must contain whatever inputs a run's policy consumes. The
> default (`TFV6_TD3_LOCAL_RESIDUAL_8_baseline_T1213`) has no `obs_speed_history`,
> so a run with `speed_history_len > 0` (e.g. `allimprovements`) needs a
> `SHARED_BUFFER` that stored it — the loader raises a clear error otherwise.

**Comparison-ready.** Register a run in `RUNS`, then pass several specs to
`plot_correction_hist([...])` to overlay runs on the shared states.

### Style

Matches the thesis figures (Computer Modern via matplotlib's bundled `cmr10`,
Paul Tol *bright* palette), identical to `result_graphs.ipynb`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# Make the repo importable regardless of where the kernel was started: walk up
# until we find the package, then put that dir on sys.path.
def _add_repo_to_path(marker="rl_finetuning"):
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / marker / "__init__.py").exists() or (d / marker).is_dir():
            if str(d) not in sys.path:
                sys.path.insert(0, str(d))
            return d
    return Path.cwd()
_REPO = _add_repo_to_path()

USE_TEX = False
SERIF = ["cmr10", "CMU Serif", "Latin Modern Roman", "STIXGeneral", "DejaVu Serif"]
PALETTE = {"blue": "#4477AA", "red": "#EE6677", "green": "#228833", "yellow": "#CCBB44",
           "cyan": "#66CCEE", "purple": "#AA3377", "grey": "#BBBBBB"}
CYCLE = [PALETTE[k] for k in ("blue", "red", "green", "purple", "cyan", "yellow")]

BASE = 11
mpl.rcParams.update({
    "text.usetex": USE_TEX,
    "font.family": "serif", "font.serif": SERIF,
    "mathtext.fontset": "cm", "axes.unicode_minus": False,
    "axes.formatter.use_mathtext": True,
    "font.size": BASE, "axes.titlesize": BASE + 1, "axes.labelsize": BASE,
    "xtick.labelsize": BASE - 1, "ytick.labelsize": BASE - 1, "legend.fontsize": BASE - 1,
    "axes.linewidth": 0.8, "axes.edgecolor": "#444444",
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": "#CCCCCC", "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "lines.linewidth": 1.8, "legend.frameon": False, "legend.handlelength": 1.6,
    "figure.figsize": (5.9, 3.6), "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.constrained_layout.use": True,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
})
print("matplotlib", mpl.__version__, "| serif ->",
      Path(mpl.font_manager.findfont("serif")).name)

### Paths & run registry

Register a residual run once (short key &rarr; folder name under
`outputs/rl_logs/`); refer to it by key everywhere afterwards. `TFV6_BASE` is the
**local** frozen TFv6 checkpoint: the run's own `config.json` points at a cluster
path, so we override it with the copy that exists on this machine.

In [ ]:
import json
from types import SimpleNamespace

def find_repo_root(marker="outputs/rl_logs"):
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / marker).exists():
            return d
    return here

REPO_ROOT = find_repo_root()
LOG_ROOT = REPO_ROOT / "outputs" / "rl_logs"
FIG_DIR = REPO_ROOT / "rl_finetuning" / "notebooks" / "figures"
CACHE_DIR = REPO_ROOT / "rl_finetuning" / "notebooks" / "cache"

# Frozen TFv6 base policy (local). The residual is added on top of this.
TFV6_BASE = REPO_ROOT / "outputs" / "checkpoints" / "tfv6_resnet34"
# Offline dataset of real frames to replay the policy over (rgb / lidar / metas).
DATA_ROOT = REPO_ROOT / "data" / "carla_leaderboard2" / "data"

RUNS = {
    "baseline": "TD3_idun_baseline",
    "allimprovements": "TD3_idun_baseline_allimprovements",
}

def run_dir(run):
    """Accept a registry key or a raw folder name."""
    return LOG_ROOT / RUNS.get(run, run)

print("repo root :", REPO_ROOT)
print("tfv6 base :", TFV6_BASE, "(exists:", TFV6_BASE.exists(), ")")
print("data root :", DATA_ROOT, "(exists:", DATA_ROOT.exists(), ")")
print("runs      :", list(RUNS))

### Model loading & correction extraction

`load_residual_actor` rebuilds the frozen TFv6 backbone + residual head and loads
a run's TD3 weights (mirroring the trainer's `_load_td3_checkpoint`).
`correction_samples` runs that actor over the **shared** pre-encoded states from
`SHARED_BUFFER` (via `forward_coeffs_from_features`, so TFv6 never re-runs) and
returns the per-frame speed correction $\Delta v$ in m/s. Results cache to
`cache/<run>__on__<SHARED_BUFFER>.npy`; pass `refresh=True` to recompute.

In [ ]:
import functools
import torch

from rl_finetuning.tfv6_rl.residual_backbone import TFv6ResidualBackbone
from rl_finetuning.tfv6_rl.policy_tfv6_td3 import TFv6ResidualActorTD3
from rl_finetuning.tfv6_rl.policy_tfv6_ppo import load_training_config

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Shared evaluation states ─────────────────────────────────────────────────
# EVERY run is evaluated on the SAME pre-encoded states, taken from this one
# buffer. That way a comparison between runs reflects only the difference in the
# policy weights, never a difference in the states they were scored on.
SHARED_BUFFER = "TFV6_TD3_VC06_RESIDUAL_6_baseline_4env_speedhistory"  # Has speed history andruns without weather.
# A few thousand states already give a clean histogram; reading every BEV map
# off disk is the slow part, so we stride-sample this many by default.
DEFAULT_FRAMES = 4000

def _rl_config(run):
    """Load the run's config.json, overriding tfv6_checkpoint with the local base."""
    cfg = json.loads((run_dir(run) / "config.json").read_text())
    cfg["tfv6_checkpoint"] = str(TFV6_BASE)
    return SimpleNamespace(**cfg)

@functools.lru_cache(maxsize=None)
def _max_speed():
    """speed_scale = TFv6 max_speed (m/s); the residual config.json omits it."""
    return float(load_training_config(str(TFV6_BASE)).max_speed)

def _alpha_speed(run):
    return float(json.loads((run_dir(run) / "config.json").read_text())["residual_alpha_speed"])

def correction_bound(run):
    """Hard limit on |delta v| in m/s: alpha_speed * v_max (coeff is tanh-bounded)."""
    return _alpha_speed(run) * _max_speed()

@functools.lru_cache(maxsize=None)
def load_residual_actor(run):
    """Return (backbone, actor) for a residual run, on DEVICE."""
    cfg = _rl_config(run)
    ckpt_file = sorted(run_dir(run).glob("model_*.pth"))[-1]
    state = torch.load(ckpt_file, map_location=DEVICE)
    backbone = TFv6ResidualBackbone(
        tfv6_checkpoint=str(TFV6_BASE), tfv6_prefix="model",
        device=DEVICE, rl_config=cfg,
    ).to(DEVICE)
    backbone.load_encoder_state_dict(state["backbone"])
    backbone.eval()
    actor = TFv6ResidualActorTD3(backbone, cfg).to(DEVICE)
    actor.residual_out.load_state_dict(state["actor_residual_out"])
    actor.eval()
    print(f"[{run}] {ckpt_file.name} | alpha_speed={backbone.residual_alpha_speed} "
          f"speed_scale={backbone.action_codec.speed_scale} "
          f"eff_rank={actor.effective_rank} route_disabled={backbone.disable_residual_route} "
          f"speed_history_len={backbone.speed_history_len}")
    return backbone, actor

@functools.lru_cache(maxsize=None)
def _shared_indices(buffer_run, n_frames):
    """Even stride-sampled row indices into the shared buffer (cached, so every
    run reuses the exact same rows)."""
    d = np.load(run_dir(buffer_run) / "buffer_latest.npz", mmap_mode="r")
    n = int(d["_capacity"][0]) if bool(d["_full"][0]) else int(d["_pos"][0])
    if n_frames is None or n_frames >= n:
        return np.arange(n)
    return np.linspace(0, n - 1, n_frames).astype(np.int64)

def correction_samples(run, *, buffer_run=SHARED_BUFFER, max_frames="default",
                       batch_size=256, refresh=False, cache=True):
    """Per-frame residual speed correction (m/s) for `run`, evaluated on the
    SHARED set of pre-encoded states from `buffer_run`'s replay buffer.

    Runs `run`'s actor via `forward_coeffs_from_features` on the stored
    `obs_bev` / `obs_kv_status_mean` / `obs_base_action_mean` (and `obs_speed_history`
    when the policy needs it), so the frozen TFv6 runs ZERO times and every run is
    scored on identical states. Returns the deterministic (noise-free) correction.

    `max_frames="default"` stride-samples DEFAULT_FRAMES rows; pass an int to cap
    or None for the whole buffer. Cached to
    cache/<run>__on__<buffer_run>[_N].npy; pass refresh=True to recompute.
    """
    n_frames = DEFAULT_FRAMES if max_frames == "default" else max_frames
    suffix = "" if max_frames == "default" else (f"_{n_frames}" if n_frames is not None else "_all")
    cache_file = CACHE_DIR / f"{run}__on__{buffer_run}{suffix}.npy"
    if cache and not refresh and cache_file.exists():
        return np.load(cache_file)

    backbone, actor = load_residual_actor(run)
    need_speed_hist = backbone.speed_history_len > 0

    bpath = run_dir(buffer_run) / "buffer_latest.npz"
    if not bpath.exists():
        raise FileNotFoundError(f"Shared buffer not found: {bpath}")
    d = np.load(bpath, mmap_mode="r")
    if need_speed_hist and "obs_speed_history" not in d.files:
        raise ValueError(
            f"{run} needs speed_history (speed_history_len={backbone.speed_history_len}), "
            f"but the shared buffer '{buffer_run}' did not store obs_speed_history. "
            f"Set SHARED_BUFFER to a run that did (e.g. one with 'speedhistory' in its "
            f"name), or pass buffer_run=... ."
        )

    idx = _shared_indices(buffer_run, n_frames)
    alpha, scale, eff = (backbone.residual_alpha_speed,
                         backbone.action_codec.speed_scale, actor.effective_rank)
    bev, kv, ba = d["obs_bev"], d["obs_kv_status_mean"], d["obs_base_action_mean"]
    sh = d["obs_speed_history"] if need_speed_hist else None
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), batch_size):
            j = idx[i:i + batch_size]
            feat = {
                "bev": torch.tensor(np.asarray(bev[j]), device=DEVICE).float(),
                "kv_status_mean": torch.tensor(np.asarray(kv[j]), device=DEVICE).float(),
                "base_action_mean": torch.tensor(np.asarray(ba[j]), device=DEVICE).float(),
            }
            if need_speed_hist:
                feat["speed_history"] = torch.tensor(np.asarray(sh[j]), device=DEVICE).float()
            coeffs = actor.forward_coeffs_from_features(feat)   # [B, eff_rank+1]
            delta = alpha * coeffs[:, eff] * scale             # m/s
            out.append(delta.detach().cpu().float().numpy().reshape(-1))
    arr = np.concatenate(out) if out else np.empty(0, np.float32)
    if cache:
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        np.save(cache_file, arr)
    print(f"[{run}] on {buffer_run} | n={len(arr)} mean={arr.mean():.3f} "
          f"std={arr.std():.3f} min={arr.min():.3f} max={arr.max():.3f} m/s")
    return arr

### Histogram

`plot_correction_hist([...])` overlays one or more runs. The dashed line at
$\Delta v = 0$ marks "no correction"; the grey span shows the hard
$\pm\alpha_{\text{speed}} v_{\max}$ bound. Pass several specs to compare runs.

In [ ]:
def save_fig(fig, name, formats=("pdf", "png")):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        fig.savefig(FIG_DIR / f"{name}.{fmt}")
    print("saved:", ", ".join(f"{name}.{f}" for f in formats), "->", FIG_DIR)

def plot_correction_hist(specs, *, bins=61, hist_range=None, density=True,
                         show_bound=True, show_mean=True,
                         xlabel=r"Residual speed correction $\Delta v$ (m/s)",
                         ylabel=None, title=None, figsize=None, save=None, **sample_kw):
    """Histogram of per-frame speed corrections for one or several runs.

    Each spec is a dict {run, label, [color]}. All runs are scored on the same
    shared buffer states (see correction_samples), so overlays differ only by the
    policy. Extra kwargs (buffer_run, max_frames, refresh, ...) go to
    correction_samples.
    """
    data = [(sp, correction_samples(sp["run"], **sample_kw)) for sp in specs]
    b = max(correction_bound(sp["run"]) for sp, _ in data)
    if hist_range is None:
        hist_range = (-b * 1.02, b * 1.02)
    if ylabel is None:
        ylabel = "Density" if density else "Count"

    fig, ax = plt.subplots(figsize=figsize)
    if show_bound:
        ax.axvspan(-b, b, color=PALETTE["grey"], alpha=0.12, lw=0, zorder=0,
                   label=r"$\pm\,\alpha_{\mathrm{speed}}\,v_{\max}$")
    ax.axvline(0.0, color="#888888", lw=0.8, ls=(0, (4, 3)), zorder=1)

    for i, (sp, arr) in enumerate(data):
        color = sp.get("color", CYCLE[i % len(CYCLE)])
        ax.hist(arr, bins=bins, range=hist_range, density=density,
                histtype="stepfilled", color=color, alpha=0.30, lw=0, zorder=2)
        ax.hist(arr, bins=bins, range=hist_range, density=density,
                histtype="step", color=color, lw=1.6, label=sp["label"], zorder=3)
        if show_mean:
            ax.axvline(arr.mean(), color=color, lw=1.3, ls=":", zorder=4)

    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_xlim(*hist_range)
    if title:
        ax.set_title(title)
    ax.legend(loc="upper left")
    if save:
        save_fig(fig, save)
    return fig, ax

## TD3 residual baseline

Distribution of the speed correction for `TD3_idun_baseline`. (First run computes
and caches; later runs load the cache instantly.)

In [ ]:
fig, ax = plot_correction_hist(
    [{"run": "baseline", "label": "TD3 Residual Baseline", "color": PALETTE["blue"]}],
    show_bound=False, show_mean=False,
    title="Residual speed-correction distribution",
    save="residual_speed_correction_baseline",
)

## Available shared buffers

`list_buffer_runs()` shows which runs saved a replay buffer big enough to use as
`SHARED_BUFFER`, and whether each stored `obs_speed_history` (needed to score
runs with `speed_history_len > 0`, such as `allimprovements`).

In [ ]:
def list_buffer_runs(min_size=1000):
    """Runs under outputs/rl_logs with a usable saved replay buffer (candidates
    for SHARED_BUFFER). Reports size and whether speed_history was stored."""
    found = []
    for p in sorted(LOG_ROOT.glob("*/buffer_latest.npz")):
        d = np.load(p, mmap_mode="r")
        n = int(d["_capacity"][0]) if bool(d["_full"][0]) else int(d["_pos"][0])
        if n >= min_size:
            found.append((p.parent.name, n, "obs_speed_history" in d.files))
    print(f"{'N':>8}  {'speed_hist':>10}  run")
    for name, n, sh in found:
        print(f"{n:>8}  {str(sh):>10}  {name}")
    return [name for name, _, _ in found]

_buffer_runs = list_buffer_runs()
print("\nSHARED_BUFFER =", SHARED_BUFFER)

## Comparing runs on identical states

Pass several specs to overlay them. Because every run is scored on the **same**
`SHARED_BUFFER` states, the only thing that differs between curves is the policy.

`allimprovements` has `speed_history_len=10`, which the default `SHARED_BUFFER`
(`...8_baseline_T1213`) did not store. To compare it against `baseline`, point
`SHARED_BUFFER` (or per-call `buffer_run=`) at a buffer that saved speed history
— `list_buffer_runs()` above flags which ones did — so **both** runs are still
scored on the same states.

In [ ]:
# Compare runs on the SAME shared states. `allimprovements` needs a SHARED_BUFFER
# that (a) stored speed_history and (b) came from a CNN run (obs_kv_status_mean,
# not the transformer's obs_kv_status_tokens), so the pre-encoded features match.
# We use the VC06 speed-history buffer: CNN-compatible, NOT random weather, and
# the same route_Town12_00 routes as the baseline. Both runs are scored on its
# identical states, so the only difference between the curves is the policy.
SPEEDHIST_BUFFER = "TFV6_TD3_VC06_RESIDUAL_6_baseline_4env_speedhistory"

fig, ax = plot_correction_hist(
    [
        {"run": "baseline",        "label": "Baseline",         "color": PALETTE["blue"]},
     {"run": "TD3_idun_baseline_regularization002", "label": r"+Regularization $\lambda^R = 0.02$", "color": PALETTE["red"]},
    #  {"run": "TD3_idun_baseline_regularization", "label": "Regularization 0.1", "color": PALETTE["green"]}
     ],
    #  {"run": "allimprovements", "label": "All improvements", "color": PALETTE["red"]}],
    show_bound=False, show_mean=False,
    buffer_run=SPEEDHIST_BUFFER,
    title="Residual speed correction: baseline vs all improvements",
    save="residual_speed_correction_comparison",
)

## Reuse &amp; extending

- **Register a run:** add it to `RUNS` (key &rarr; folder under `outputs/rl_logs/`)
  or pass the raw folder name as `run`. It only needs a `model_*.pth` checkpoint.
- **Change the shared states:** set `SHARED_BUFFER` (global) or pass
  `buffer_run=...` per call. All runs in one figure are scored on that buffer, so
  comparisons stay apples-to-apples.
- **Sample size:** `max_frames=N` stride-samples N states (default
  `DEFAULT_FRAMES=4000`); `max_frames=None` uses the whole buffer; `refresh=True`
  recomputes and overwrites the cache.
- **Histogram look:** `plot_correction_hist` takes `bins`, `hist_range`,
  `density` (`True` keeps shapes comparable across runs of different size),
  `show_bound`, `show_mean`.

Caveats: corrections are the **deterministic** actor output (no exploration
noise), evaluated on the shared buffer's state distribution (that buffer's
policy's states, not each run's own). `SHARED_BUFFER` must contain every input a
run's policy consumes (e.g. `obs_speed_history`); the loader errors clearly if
not.